# 37 — Bullet Parsing & STAR Scoring
**Goal:** Score resume bullets on action verbs, metrics, and STAR compliance.

A resume bullet is a mini-argument: "Reduced model latency by 40% by optimizing inference pipeline". The STAR method (Situation, Task, Action, Result) is the rubric recruiters use to judge whether that argument is strong. This chapter scores bullets mechanically — action verbs, quantified metrics, context, outcome — with pure rules, no LLM.

**Why it matters for resumes / ATS:** screening systems rank candidates partly on bullet quality. Rule-based STAR scoring gives an instant, consistent, explainable 0–1 quality signal per bullet — the feature that lets an ATS (or a career coach) say "this bullet is weak, add a metric" — and it is the same signal Ch. 39 stores per bullet for downstream reporting.

## 1. STAR Method Explained

STAR = **S**ituation, **T**ask, **A**ction, **R**esult. A strong bullet leads with a strong action verb, quantifies the result, names the technology or context, and states an outcome. A weak one is passive ("Was responsible for ML models"), metric-free, and result-free.

**What the code does:** the cell is a printed reference card. It contrasts a good bullet ("Reduced model latency by 40% by optimizing inference pipeline" — Action `Reduced`, Result `40%`, Task `optimizing inference`) against a weak one, then lists the four scoring criteria the rest of the chapter encodes: strong action verb, quantified metric, specific context, result/outcome.

**Why it matters:** these four criteria are *operationalizable* — each maps to a regex check in the next cells. That is the whole trick of rule-based scoring: turn a hiring heuristic into a checklist.

In [ ]:
print('''STAR = Situation, Task, Action, Result
Good bullet (STAR compliant):
  "Reduced model latency by 40% by optimizing inference pipeline"
  Action: Reduced | Result: 40% | Task: optimizing inference

Weak bullet (no STAR):
  "Was responsible for ML models"
  → Passive voice, no metric, no result

Scoring criteria:
  ✓ Strong action verb (Developed, Led, Reduced, Built)
  ✓ Quantified metric (% improvement, $, count)
  ✓ Specific context (technology, team size)
  ✓ Result/outcome mentioned''')

## 2. Bullet Scorer

`score_bullet()` converts the STAR rubric into a 0–1 score by summing weighted regex checks, capped at 1.0:

| Check | Weight |
|---|---|
| First word is a strong action verb (21 verbs in `ACTION_VERBS`) | +0.30 |
| Quantified metric (`%`, `million`, `$`, `x`, `percent`) | +0.30 |
| "by N" improvement phrasing | +0.15 |
| Technology context (`using`/`with`/`via` + capital letter) | +0.15 |
| Context nouns (`team`, `pipeline`, `system`, `platform`) | +0.10 |

**Verified on the five test bullets:** the TensorFlow bullet scores `0.75` (STRONG); "Led team of 5 engineers to deliver ML platform" scores `0.40` (GOOD); "Improved customer engagement with data-driven recommendations" scores `0.30` (WEAK) — an action verb with no metric to back it; "Was responsible for ML model development" and "Worked on various projects" score `0.00` (WEAK).

**Try it:** the weights are a judgment call, not gospel — rebalance them and the STRONG/GOOD/WEAK boundaries move with your hiring priorities.

In [ ]:
import re

ACTION_VERBS = {"developed", "led", "reduced", "built", "designed", "implemented",
    "created", "managed", "delivered", "achieved", "improved", "increased",
    "decreased", "optimized", "architected", "spearheaded", "established",
    "launched", "generated", "transformed", "engineered"}

def score_bullet(bullet):
    """Score a single resume bullet 0-1."""
    text = bullet.strip()
    score = 0
    
    # Check starts with action verb
    first_word = text.split()[0].lower().rstrip(",.;:")
    if first_word in ACTION_VERBS:
        score += 0.3
    
    # Check for quantified metrics
    if re.search(r"\d+\s*(%|million|billion|\$|x|percent)", text, re.IGNORECASE):
        score += 0.3
    if re.search(r"\bby\s+\d+", text, re.IGNORECASE):
        score += 0.15
    
    # Check for specific context
    if re.search(r"(using|with|via)\s+[A-Z]", text):
        score += 0.15
    if any(word in text.lower() for word in ["team", "pipeline", "system", "platform"]):
        score += 0.1
    
    return min(score, 1.0)

bullets = [
    "Reduced model latency by 40% through TensorFlow optimization",
    "Was responsible for ML model development",
    "Led team of 5 engineers to deliver ML platform",
    "Improved customer engagement with data-driven recommendations",
    "Worked on various projects",
]

for b in bullets:
    s = score_bullet(b)
    print(f"  {'WEAK' if s < 0.4 else 'GOOD' if s < 0.7 else 'STRONG'} ({s:.2f}) '{b[:50]}'")

## 3. STAR Compliance Check

Scoring is continuous; compliance is boolean. `star_compliance()` answers five yes/no questions about a bullet — starts with an action verb, contains a number, mentions a technology (any capitalized word), signals an outcome ("by", "resulting", "achieving", "increasing", "reducing"), and avoids passive openers ("was"/"were"/"had"/"has been").

**What the code does:** returns a dict of five booleans; the cell prints `passed/5` plus a tick per check.

**Verified on the test bullets:** the TensorFlow bullet passes all 5; "Led team of 5 engineers to deliver ML platform" passes 4 (no outcome signal); "Was responsible for ML model development" passes just 1 — `has_technology`, and only because "Was" is a capitalized word. That exposes the crude proxy: `has_technology` fires on *any* capital-letter sequence, so a passive opener alone satisfies it.

**Try it:** compare the 0.40-scoring "Led team..." bullet (4/5 STAR) with the 0.30 "Improved..." bullet (3/5) — the score and the compliance count agree on ranking but measure different things.

In [ ]:
def star_compliance(bullet):
    checks = {
        "has_action_verb": bool(re.search(r"^(" + "|".join(ACTION_VERBS) + r")", bullet, re.IGNORECASE)),
        "has_metric": bool(re.search(r"\d+", bullet)),
        "has_technology": bool(re.search(r"[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*", bullet)),
        "has_outcome": bool(re.search(r"(by|resulting|achieving|increasing|reducing)", bullet, re.IGNORECASE)),
        "is_active_voice": not bool(re.search(r"^was|^were|^had|^has been", bullet, re.IGNORECASE)),
    }
    return checks

for b in bullets:
    c = star_compliance(b)
    passed = sum(1 for v in c.values() if v)
    print(f"  {passed}/5 STAR: '{b[:45]}'")
    for check, result in c.items():
        print(f"    {check:20s}: {'✓' if result else ' '}")

## Summary: Bullet scoring identifies weak bullets for improvement. Rule-based, no LLM needed.

**Bullet quality is measurable — action verb + metric + context + outcome, summed and capped.**

The scorer and the compliance checker are two views of the same rubric: a continuous 0–1 score for ranking, and a five-flag report for coaching ("add a metric", "switch to active voice"). Both are pure regex over one line of text — fast enough to run over a whole resume in microseconds, and fully explainable to a candidate.

The `star_score` and boolean flags computed here are stored per bullet in Ch. 39's `ExperienceBullet` model, giving the final resume JSON a per-bullet quality signal. Next, Ch. 38 extracts the projects section, whose bullets get scored the same way.